# Tutorial 03 — MS-PRS Modulator and BCJR Equalizer

MS-PRS demultiplexes the bit stream into two parallel sub-streams and passes them through FIR filters `h_0` (length `L_0`) and `h_1` (length 1). The output is a real-valued symbol carrying 2 information bits.

$$ s[k] = \sqrt{\eta_0}\sum_{j=0}^{L_0-1} h_0[j]\,x^{(0)}[k-j] + \sqrt{\eta_1}\,h_1[0]\,x^{(1)}[k] $$

The receiver knows `h_0, h_1` and runs a BCJR over a `2^{L_0-1}`-state trellis.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from nsm.modem.msprs import precompute, modulate, demodulate, uncoded_ber
from nsm.channel.awgn import transmit

## Inspect the trellis

In [ ]:
L0 = 3
p  = precompute(L0, input_length=1000, filter_type='unbalanced')
print('h0:', p['h0'])
print('h1:', p['h1'])
print('memory:', p['memory'], '| states:', p['total_states'])
print('unique branch labels (4-ASK-like):', np.unique(np.round(p['branch_labels'], 3)))

## Constellation
Four distinct output amplitudes per state, but the full trellis output spans the union over states. With `L_0 = 3` it's the same five levels shown in the paper's Fig. 2.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 1.6))
ax.scatter(p['branch_labels'], np.zeros_like(p['branch_labels']),
           s=30, alpha=0.6)
ax.set_yticks([]); ax.set_xlabel('amplitude')
ax.set_title(f'MS-PRS branch labels, L0={L0} unbalanced')
plt.tight_layout()

## Uncoded BER at a single SNR

In [ ]:
src_bits = 4096
p = precompute(L0, src_bits, 'unbalanced')
eb_no_db = 10.0
eb_no_lin = 10 ** (eb_no_db / 10)
noise_var = 0.5 / (2 * eb_no_lin)
noise_std = np.sqrt(noise_var)
errs = uncoded_ber(src_bits, L0, p['h0'], p['h1'], p['branch_labels'],
                   p['memory'], p['total_states'], p['next_states'],
                   p['branch_indices'], noise_var, noise_std)
print(f'Uncoded BER at {eb_no_db} dB Eb/N0: {errs/src_bits:.4f} ({int(errs)} / {src_bits})')

## BER vs Eb/N0 sweep

In [ ]:
snrs = np.arange(0, 16.1, 2.0)
ber = []
for snr in snrs:
    nv = 0.5 / (2 * 10 ** (snr / 10)); ns = np.sqrt(nv)
    total = 0
    for _ in range(3):
        total += int(uncoded_ber(src_bits, L0, p['h0'], p['h1'], p['branch_labels'],
                                  p['memory'], p['total_states'], p['next_states'],
                                  p['branch_indices'], nv, ns))
    ber.append(total / (3 * src_bits))
plt.semilogy(snrs, np.clip(ber, 1e-5, 1), 'o-')
plt.xlabel('Eb/N0 (dB)'); plt.ylabel('BER'); plt.grid(True, which='both', alpha=0.3)
plt.title(f'Uncoded MS-PRS, L0={L0}, unbalanced')